# European Parliament data (Denmark)

Fetch EP roll-call votes for Danish MEPs, split into **folders per election period** (~5 years).

| Folder | EP term | Source |
|--------|---------|--------|
| `2009-2014` | 7th | [VoteWatch / EUI](https://hdl.handle.net/1814/74918) |
| `2014-2019` | 8th | VoteWatch |
| `2019-2024` | 9th | [HowTheyVote](https://github.com/HowTheyVote/data) |
| `2024-2029` | 10th | HowTheyVote |

Each folder contains `ep_members_dk.csv`, `ep_member_votes_dk.csv`, `ep_votes.csv`, `ep_group_memberships_dk.csv`.

National↔EP mapping is done separately against your Danish dataset.

In [ ]:
from pathlib import Path
import json
import time
import urllib.request
import pandas as pd

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

EP_API = "https://data.europarl.europa.eu/api/v2"
HTV_BASE = "https://github.com/HowTheyVote/data/releases/latest/download"

MAIN_VOTES_ONLY = True  # HowTheyVote: set False for all roll-calls

# HowTheyVote only has 9th and 10th EP (from July 2019)
EP_TERMS_HTV = {
    9: {"folder": "2019-2024", "start": "2019-07-02"},
    10: {"folder": "2024-2029", "start": "2024-07-16"},
}

# All period folders for summary
PERIOD_FOLDERS = ["2009-2014", "2014-2019", "2019-2024", "2024-2029"]

In [5]:
import urllib.error


def ep_get(path: str) -> dict:
    url = f"{EP_API}/{path.lstrip('/')}"
    for attempt in range(5):
        try:
            req = urllib.request.Request(url, headers={"Accept": "application/ld+json"})
            with urllib.request.urlopen(req) as resp:
                return json.loads(resp.read())
        except urllib.error.HTTPError as exc:
            if exc.code == 429:
                time.sleep(30 * (attempt + 1))
            else:
                raise
    raise RuntimeError("EP API rate limit exceeded (429)")


ORG_LABELS: dict[str, str] = {}


def org_label(org_id: str | None, lang: str = "da") -> str | None:
    if not org_id:
        return None
    oid = org_id.replace("org/", "")
    if oid in ORG_LABELS:
        return ORG_LABELS[oid]
    body = ep_get(f"corporate-bodies/{oid}")
    data = body.get("data", body)
    if isinstance(data, list):
        data = data[0]
    pref = data.get("prefLabel", {})
    label = pref.get(lang) or pref.get("en") or data.get("label") or oid
    ORG_LABELS[oid] = label
    return label


def load_htv_csv(name: str) -> pd.DataFrame:
    url = f"{HTV_BASE}/{name}.csv.gz"
    return pd.read_csv(url, compression="gzip")


def term_period_dir(term: int) -> Path:
    path = DATA_DIR / EP_TERMS_HTV[term]["folder"]
    path.mkdir(parents=True, exist_ok=True)
    return path


def votes_for_term(votes_df: pd.DataFrame, term: int) -> pd.DataFrame:
    """Assign votes to an EP period by plenary date."""
    start = pd.Timestamp(EP_TERMS_HTV[term]["start"], tz="UTC")
    ts = votes_df["timestamp"]
    mask = ts >= start
    later_terms = [t for t in EP_TERMS_HTV if t > term]
    if later_terms:
        next_start = min(pd.Timestamp(EP_TERMS_HTV[t]["start"], tz="UTC") for t in later_terms)
        mask &= ts < next_start
    return votes_df[mask].copy()


def save_period_data(
    term: int,
    members: pd.DataFrame,
    member_votes: pd.DataFrame,
    votes: pd.DataFrame,
    group_memberships: pd.DataFrame,
    meps: pd.DataFrame | None = None,
) -> None:
    out = term_period_dir(term)
    term_votes = votes_for_term(votes, term)
    vote_ids = set(term_votes["id"].astype(int))

    term_member_votes = member_votes[member_votes["vote_id"].isin(vote_ids)].copy()
    active_ids = set(term_member_votes["member_id"].astype(int))

    term_members = members[members["id"].astype(int).isin(active_ids)].copy()
    term_groups = group_memberships[group_memberships["term"] == term].copy()
    term_meps = (
        meps[meps["mep_id"].astype(int).isin(active_ids)].copy() if meps is not None else None
    )

    term_members.to_csv(out / "ep_members_dk.csv", index=False)
    term_member_votes.to_csv(out / "ep_member_votes_dk.csv", index=False)
    term_votes.to_csv(out / "ep_votes.csv", index=False)
    term_groups.to_csv(out / "ep_group_memberships_dk.csv", index=False)
    if term_meps is not None:
        term_meps.to_csv(out / "ep_meps_dk.csv", index=False)

    print(
        f"{EP_TERMS_HTV[term]['folder']}: "
        f"{len(term_members)} MEPs, {len(term_votes)} votes, "
        f"{len(term_member_votes)} member-vote rows"
    )

## 1. Danish MEP profiles (EP Open Data API)

Fetches every Danish MEP ID from HowTheyVote (not only the current 15). Historical MEPs may have ended national-party memberships in the API response.

In [6]:
members_preview = load_htv_csv("members")
dk_member_ids_preview = set(
    members_preview.loc[members_preview["country_code"] == "DNK", "id"].astype(int)
)

current = ep_get("meps/show-current?limit=800")["data"]
current_dk_ids = {
    int(m["identifier"])
    for m in current
    if m.get("api:country-of-representation") == "DK"
}

rows = []
for mep_id in sorted(dk_member_ids_preview):
    detail = ep_get(f"meps/{mep_id}")["data"][0]
    is_current = mep_id in current_dk_ids

    eu_org, nat_org = None, None
    for mem in detail.get("hasMembership", []):
        if mem.get("memberDuring", {}).get("endDate"):
            continue
        cls = str(mem.get("membershipClassification", ""))
        org = mem.get("organization", "")
        if "EU_POLITICAL_GROUP" in cls:
            eu_org = org
        if "NATIONAL_POLITICAL_GROUP" in cls:
            nat_org = org

    email = detail.get("hasEmail") or ""
    if isinstance(email, list):
        email = email[0] if email else ""

    rows.append(
        {
            "mep_id": mep_id,
            "name": detail.get("label") or detail.get("sortLabel"),
            "is_current": is_current,
            "ep_group": org_label(eu_org),
            "national_party": org_label(nat_org),
            "national_party_code": org_label(nat_org, lang="en"),
            "email": str(email).replace("mailto:", ""),
            "source": "ep_api",
        }
    )
    time.sleep(0.3)  # EP API: max 500 requests / 5 min

ep_meps_dk = pd.DataFrame(rows)
print(
    f"Fetched {len(ep_meps_dk)} DK MEP profiles "
    f"({ep_meps_dk['is_current'].sum()} currently in office)"
)
ep_meps_dk

Saved 15 current DK MEPs


,mep_id,name,ep_group_code,ep_group,national_party,national_party_code,email,source
0,197558,Asger CHRISTENSEN,Renew,Gruppen Renew Europe,"Venstre, Danmarks Liberale Parti","Venstre, Danmarks Liberale Parti",asger.christensen@europarl.europa.eu,ep_api_current
1,37312,Christel SCHALDEMOSE,S&D,Gruppen for Det Progressive Forbund af Sociald...,Socialdemokratiet,Socialdemokratiet,christel.schaldemose@europarl.europa.eu,ep_api_current
2,257028,Henrik DAHL,PPE,Det Europæiske Folkepartis Gruppe (Kristelige ...,Liberal Alliance,Liberal Alliance,henrik.dahl@europarl.europa.eu,ep_api_current
3,197573,Kira Marie PETER-HANSEN,Verts/ALE,Gruppen De Grønne/Den Europæiske Fri Alliance,Socialistisk Folkeparti,Socialistisk Folkeparti,kira.peter-hansen@europarl.europa.eu,ep_api_current
4,257025,Kristoffer STORM,ECR,De Europæiske Konservative og Reformister,Danmarksdemokraterne,Danmarksdemokraterne,kristoffer.storm@europarl.europa.eu,ep_api_current
5,277178,Majbritt BIRKHOLM,PfE,Gruppen Patrioter for Europa,Dansk Folkeparti,Dansk Folkeparti,majbritt.birkholm@europarl.europa.eu,ep_api_current
6,199941,Marianne VIND,S&D,Gruppen for Det Progressive Forbund af Sociald...,Socialdemokratiet,Socialdemokratiet,marianne.vind@europarl.europa.eu,ep_api_current
7,96709,Morten LØKKEGAARD,Renew,Gruppen Renew Europe,"Venstre, Danmarks Liberale Parti","Venstre, Danmarks Liberale Parti",morten.lokkegaard@europarl.europa.eu,ep_api_current
8,101585,Niels FUGLSANG,S&D,Gruppen for Det Progressive Forbund af Sociald...,Socialdemokratiet,Socialdemokratiet,niels.fuglsang@europarl.europa.eu,ep_api_current
9,257033,Niels Flemming HANSEN,PPE,Det Europæiske Folkepartis Gruppe (Kristelige ...,Det Konservative Folkeparti,Det Konservative Folkeparti,niels.hansen@europarl.europa.eu,ep_api_current


## 2. All Danish MEPs in HowTheyVote (current + past terms)

In [7]:
members = load_htv_csv("members")
ep_members_dk = members[members["country_code"] == "DNK"].copy()

dk_member_ids = set(ep_members_dk["id"].astype(int))
print(f"Loaded {len(ep_members_dk)} Danish MEPs (all terms in export)")
ep_members_dk.head()

Saved 27 Danish MEPs (all terms in export)


,id,first_name,last_name,country_code,date_of_birth,email,facebook,twitter
60,28161,Margrete,AUKEN,DNK,NaN,NaN,NaN,NaN
101,37312,Christel,SCHALDEMOSE,DNK,1967-08-04,christel.schaldemose@europarl.europa.eu,https://www.facebook.com/ChristelSchaldemose,https://twitter.com/SchaldemoseMEP
132,96709,Morten,LØKKEGAARD,DNK,NaN,morten.lokkegaard@europarl.europa.eu,https://www.facebook.com/morten.lokkegaard,http://twitter.com/loekkegaard_mep
206,101585,Niels,FUGLSANG,DNK,1985-06-29,niels.fuglsang@europarl.europa.eu,https://www.facebook.com/FuglsangEP19,https://twitter.com/NielsFuglsang
214,111412,Villy,SØVNDAL,DNK,1952-04-04,villy.sovndal@europarl.europa.eu,NaN,NaN


## 3. Roll-call votes (DK MEP positions only)

Downloads `member_votes.csv.gz` (~65 MB) and keeps rows for Danish MEP IDs only.

In [8]:
member_votes = load_htv_csv("member_votes")
member_votes["member_id"] = member_votes["member_id"].astype(int)

ep_member_votes_dk = member_votes[member_votes["member_id"].isin(dk_member_ids)].copy()

print(f"Loaded {len(ep_member_votes_dk):,} member-vote rows")
print(f"  unique MEPs: {ep_member_votes_dk['member_id'].nunique()}")
print(f"  unique votes: {ep_member_votes_dk['vote_id'].nunique()}")
ep_member_votes_dk.head()

Saved 352,614 member-vote rows
  unique MEPs: 26
  unique votes: 24844


,vote_id,member_id,position,country_code,group_code
49,108425,28161,FOR,DNK,GREEN_EFA
83,108425,37312,ABSTENTION,DNK,SD
107,108425,96709,DID_NOT_VOTE,DNK,RENEW
175,108425,101585,DID_NOT_VOTE,DNK,SD
255,108425,124872,DID_NOT_VOTE,DNK,RENEW


## 4. Vote metadata (all periods, then split by folder in step 6)

In [9]:
votes = load_htv_csv("votes")
vote_ids = set(ep_member_votes_dk["vote_id"].astype(int))

ep_votes = votes[votes["id"].isin(vote_ids)].copy()
ep_votes["timestamp"] = pd.to_datetime(ep_votes["timestamp"], utc=True, format="mixed")

if MAIN_VOTES_ONLY and "is_main" in ep_votes.columns:
    ep_votes = ep_votes[ep_votes["is_main"] == True]

kept_vote_ids = set(ep_votes["id"].astype(int))
ep_member_votes_dk = ep_member_votes_dk[
    ep_member_votes_dk["vote_id"].isin(kept_vote_ids)
].copy()

print(f"Loaded {len(ep_votes):,} votes (main_only={MAIN_VOTES_ONLY})")
print(f"  member-vote rows: {len(ep_member_votes_dk):,}")
print(f"  date range: {ep_votes['timestamp'].min()} -> {ep_votes['timestamp'].max()}")
ep_votes.head()

Saved 614 votes (term 10+, main_only=True)
  date range: 2024-07-17 12:50:14+00:00 -> 2026-07-09 12:46:04+00:00


,id,timestamp,display_title,reference,description,amendment_subject,amendment_number,is_main,procedure_reference,procedure_title,procedure_type,procedure_stage,count_for,count_against,count_abstention,count_did_not_vote,result,texts_adopted_reference
19297,169362,2024-07-17 12:50:14+00:00,The need for the EU's continuous support for U...,B10-0007/2024,Proposition de résolution (ensemble du texte),NaN,NaN,True,2024/2721(RSP),The need for the EU’s continuous support for U...,RSP,NaN,495,137,47,40,ADOPTED,P10_TA(2024)0003
19301,169401,2024-09-19 12:13:40+00:00,EU/USA Agreement on the launch of Galileo sate...,A10-0001/2024,Projet de décision du Conseil,NaN,NaN,True,2024/0046(NLE),EU/USA Agreement: setting forth security proce...,NLE,NaN,582,16,17,102,ADOPTED,P10_TA(2024)0011
19304,169418,2024-09-18 12:18:27+00:00,"Objection pursuant to Rule 115(2) and (3), and...",B10-0020/2024,Proposition de résolution,NaN,NaN,True,2024/2758(RPS),The draft Commission regulation amending Annex...,RPS,NaN,516,129,27,45,ADOPTED,P10_TA(2024)0006
19305,169419,2024-09-18 12:18:58+00:00,"Objection pursuant to Rule 115(2) and (3), and...",B10-0021/2024,Proposition de résolution,NaN,NaN,True,2024/2759(RPS),The draft Commission regulation amending Annex...,RPS,NaN,522,127,28,40,ADOPTED,P10_TA(2024)0007
19313,169541,2024-09-19 12:13:11+00:00,The case of José Daniel Ferrier García in Cuba,RC-B10-0022/2024,Proposition de résolution (ensemble du texte),NaN,NaN,True,2024/2805(RSP),The case of José Daniel Ferrer García in Cuba,RSP,NaN,380,182,51,104,ADOPTED,P10_TA(2024)0010


## 5. EP political group memberships (time-varying)

In [10]:
groups = load_htv_csv("groups")
group_memberships = load_htv_csv("group_memberships")
group_memberships["member_id"] = group_memberships["member_id"].astype(int)

ep_group_memberships_dk = group_memberships[
    group_memberships["member_id"].isin(dk_member_ids)
].copy()
ep_group_memberships_dk = ep_group_memberships_dk.merge(
    groups[["code", "label", "short_label"]],
    left_on="group_code",
    right_on="code",
    how="left",
)

print(f"Loaded {len(ep_group_memberships_dk)} group membership rows")
ep_group_memberships_dk.head()

Saved 37 group membership rows


,member_id,group_code,term,start_date,end_date,code,label,short_label
0,28161,GREEN_EFA,9,2019-07-02,2024-07-15,GREEN_EFA,Greens/European Free Alliance,Greens/EFA
1,37312,SD,9,2019-07-02,2024-07-15,SD,Progressive Alliance of Socialists and Democrats,S&D
2,37312,SD,10,2024-07-16,NaN,SD,Progressive Alliance of Socialists and Democrats,S&D
3,96709,RENEW,9,2019-07-02,2024-07-15,RENEW,Renew Europe,Renew
4,96709,RENEW,10,2024-07-16,NaN,RENEW,Renew Europe,Renew


## 6a. Terms 7–8 from VoteWatch (2009–2019)

HowTheyVote does not include these periods. We use the [VoteWatch archive](https://hdl.handle.net/1814/74918) (Simon Hix et al.).

**Note:** EP7 uses internal VoteWatch IDs; EP8 uses europarl website IDs where available. Link people by name across periods.

In [ ]:
from fetch_votewatch_terms import process_votewatch_term

for term in (7, 8):
    process_votewatch_term(term)

## 6b. Terms 9–10 from HowTheyVote (2019–2029)

In [ ]:
for term in sorted(EP_TERMS_HTV):
    save_period_data(
        term,
        ep_members_dk,
        ep_member_votes_dk,
        ep_votes,
        ep_group_memberships_dk,
        ep_meps_dk,
    )

## Summary (per period)

In [11]:
for folder_name in PERIOD_FOLDERS:
    folder = DATA_DIR / folder_name
    if not folder.exists():
        continue
    print(f"\n{folder_name}")
    for path in sorted(folder.glob("ep_*.csv")):
        df = pd.read_csv(path)
        print(f"  {path.name:30} {len(df):>8,} rows")

ep_group_memberships_dk.csv          37 rows
ep_member_votes_dk.csv            9,210 rows
ep_members_dk.csv                    27 rows
ep_meps_dk.csv                       15 rows
ep_votes.csv                        614 rows
